In [2]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import oracledb
import sys
import time
from datetime import datetime

# 获取当前时间戳
# current_time = time.time()

# 将时间戳转换为 datetime 对象
formatted_time = datetime.fromtimestamp(time.time())

hour = formatted_time.hour
minute = formatted_time.minute
second = formatted_time.second


In [15]:
yea = formatted_time.year
mon = formatted_time.month
day = formatted_time.day

In [17]:
print(f'{yea}-{mon}-{day}-{hour}-{minute}')

2024-1-24-4-50


In [ ]:
from mod_functions import mod_load_config,send_msg
mod_config = mod_load_config()

In [24]:
print(f"{hour}+{minute}+{second}")
print(f"Formatted time: {hour:02d}:{minute:02d}:{second:02d}")

connection=oracledb.connect(
     config_dir= mod_config["wallet_dir"],
     user=mod_config["oracle_user"],
     password=mod_config["oracle_password"],
     dsn = mod_config["dsn"],
     wallet_location=mod_config["wallet_location"],
     wallet_password=mod_config["wallet_password"])
cursor = connection.cursor()
tn = mod_config["table_name"]
sql = f"SELECT TO_CHAR(ut, 'YYYY-MM-DD HH24:MI:SS') AS formatted_ut, symList FROM {tn}"
cursor.execute(sql)
df = pd.read_sql(sql,connection)
connection.close()


4+50+13
Formatted time: 04:50:13


C:\Users\Neo\AppData\Local\Temp\ipykernel_16984\3438081481.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql,connection)


In [4]:

df["SYMLIST"] = df["SYMLIST"].transform(eval)

symPool = [i[1] for sublist in df['SYMLIST'].values for i in sublist]
unique_values = np.unique(symPool).tolist()

dfhtmp = pd.DataFrame(columns=['timestamp'] + unique_values)
dfhtmp["timestamp"] = df["FORMATTED_UT"]
dfhtmp["timestamp"] = pd.to_datetime(df["FORMATTED_UT"])


In [5]:

cols = dfhtmp.columns
inum = 1
while inum < len(cols) :
    sym = cols[inum]
    for index, row in df.iterrows():
        for pair in row[1]:
            if pair[1] == sym :
                dfhtmp.iloc[index,inum] = pair[0]
            # else:
            #     if dfhtmp.iloc[index,inum] == "NaN":
            #         dfhtmp.iloc[index,inum] = 0
    inum += 1 
    
dfhtmp.set_index("timestamp", inplace=True)
dfhtmp = dfhtmp.sort_values(by='timestamp')
dfhtmp = dfhtmp.fillna(0.0)


In [20]:
ddf= dfhtmp.describe()
ddfstd = ddf.sort_values(by='mean',axis='columns')

htmp_tmstmp = f"{yea}-{mon}-{day}-{hour}-{minute}"
htmp_title = f"HOTMAP  {htmp_tmstmp}"


In [ ]:

plt.title(htmp_title)
sns.color_palette("mako", as_cmap=True)


In [19]:
plt.figure(figsize=(100, 40))


<Figure size 10000x4000 with 0 Axes>

<Figure size 10000x4000 with 0 Axes>

In [ ]:

dfhtmp_reordered = dfhtmp.reindex(columns=ddfstd.columns)
dfhtmp_ds = dfhtmp_reordered.describe()

df_min = dfhtmp_ds.loc['max'][dfhtmp_ds.loc['max'] != 0].min()
df_max = dfhtmp_ds.loc['max'].max()

sns.heatmap(dfhtmp_reordered.T, annot=False, fmt=".6f", vmin = df_min, vmax = df_max, linewidths=.05, cbar_kws={'label': 'Your Colorbar Label'})
plt.savefig(f"hotmap-{htmp_tmstmp}.png")  # 保存为PNG格式
# script_name = sys.argv[0]
# arguments = sys.argv[1:]
print(f"HOTMAP  {yea}-{mon}-{day}-{hour}-{minute}.png SAVED ")
# plt.savefig(arguments[0])  # 保存为PNG格式
# plt.show()